# RAR Stage-4 LAION-Aesthetics Audit — Kaggle / Colab notebook

Companion to *We Are Trapped in RAR: How Human Selection Hardens Generative AI Bias* (Acharjee Porag, Nowshin, Himel, 2026).

**What this does:** samples images from the unfiltered LAION-2B baseline and the LAION-Aesthetics-V2 high-score subset, classifies perceived demographics with off-the-shelf models, and tests whether the aesthetic-filtered subset is more demographically stereotyped per occupation than baseline (RAR Stage-4 hypothesis H1).

**Before running you need:**
1. A Hugging Face token with read access (`huggingface.co/settings/tokens`).
2. *Granted* access to:
   - `laion/relaion2B-en-research-safe`
   - `laion/laion2B-en-aesthetic`
   - Both are gated; request via the dataset card. Approval is typically auto for academic use, 1–2 days.
3. On Kaggle: store your HF token in **Add-ons → Secrets** as `HF_TOKEN`.
   On Colab: paste it into the cell below.

**Two run modes (set in the config cell below):**
- `SMOKE = True` — leadership category only, 50 images per subset per occupation. ~30 min. Run this first to confirm the pipeline works.
- `SMOKE = False` — all 15 occupations, 1000 images per subset per occupation, three aesthetic thresholds. ~6–8 hours on a Kaggle P100. **This produces the numbers for Section 5 of the paper.**

## Setup

In [ ]:
%%capture
!pip install -q datasets==2.21.0 huggingface_hub Pillow numpy scipy pandas matplotlib requests tqdm
!pip install -q deepface tf-keras  # tf-keras needed for newer TF compat with deepface

In [ ]:
import os, sys

# --- HF auth -------------------------------------------------------------
# Kaggle: store as Secret named HF_TOKEN.
# Colab: replace the literal string below, OR use the Secrets sidebar.
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient  # type: ignore
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token loaded from Kaggle Secrets.')
except Exception:
    try:
        from google.colab import userdata  # type: ignore
        HF_TOKEN = userdata.get('HF_TOKEN')
        print('HF token loaded from Colab Secrets.')
    except Exception:
        # Last resort: paste here for one-off runs. DON'T COMMIT.
        HF_TOKEN = os.environ.get('HF_TOKEN') or ''
        if HF_TOKEN:
            print('HF token loaded from environment.')
        else:
            print('!! HF_TOKEN not set — gated datasets will fail. Set it before continuing.')

os.environ['HF_TOKEN'] = HF_TOKEN or ''

## Configuration

In [ ]:
# --- Run mode ------------------------------------------------------------
SMOKE = True  # set False for the full paper-quality run

if SMOKE:
    OCCUPATION_CATEGORIES = ['leadership']
    SAMPLES_PER_SUBSET = 50
    AESTHETIC_THRESHOLDS = (6.0,)
else:
    OCCUPATION_CATEGORIES = ['leadership', 'care', 'stem']
    SAMPLES_PER_SUBSET = 1000
    AESTHETIC_THRESHOLDS = (5.0, 6.0, 6.5)

# Output dir — Kaggle persists /kaggle/working/, Colab needs Drive mount.
OUT = '/kaggle/working' if os.path.exists('/kaggle/working') else './rar_results'
os.makedirs(OUT, exist_ok=True)
print(f'Mode: {"SMOKE" if SMOKE else "FULL"}')
print(f'Categories: {OCCUPATION_CATEGORIES}')
print(f'Samples per subset per occupation: {SAMPLES_PER_SUBSET}')
print(f'Aesthetic thresholds: {AESTHETIC_THRESHOLDS}')
print(f'Output: {OUT}')

In [ ]:
OCCUPATIONS = {
    'leadership': [
        ('ceo',       r'\bceo\b|chief\s+executive'),
        ('founder',   r'\bfounder\b'),
        ('manager',   r'\bmanager\b'),
        ('leader',    r'\bleader\b'),
        ('executive', r'\bexecutive\b'),
    ],
    'care': [
        ('nurse',         r'\bnurse\b'),
        ('teacher',       r'\bteacher\b'),
        ('caregiver',     r'\bcaregiver\b|care\s*giver'),
        ('receptionist',  r'\breceptionist\b'),
        ('social_worker', r'social\s+worker'),
    ],
    'stem': [
        ('scientist',  r'\bscientist\b'),
        ('engineer',   r'\bengineer\b'),
        ('doctor',     r'\bdoctor\b'),
        ('researcher', r'\bresearcher\b'),
        ('programmer', r'\bprogrammer\b|software\s+developer'),
    ],
}

DATASETS = {
    'baseline':   'laion/relaion2B-en-research-safe',
    'aesthetics': 'laion/laion2B-en-aesthetic',
}

## Sampling

In [ ]:
import re, json, time
from dataclasses import dataclass, asdict, field
from collections import Counter
from datasets import load_dataset
from tqdm.auto import tqdm

@dataclass
class Sample:
    occupation: str
    subset: str
    image_url: str
    caption: str
    aesthetic_score: float | None = None
    perceived_gender: str | None = None
    perceived_race: str | None = None
    color_hist: list = field(default_factory=list)

def caption_matches(caption, regex):
    return bool(re.search(regex, caption or '', flags=re.IGNORECASE))

def stream_samples(dataset_id, occupation, regex, target_n, *, aesthetic_min=None, hf_token=None):
    ds = load_dataset(dataset_id, split='train', streaming=True, token=hf_token)
    yielded = 0
    seen = 0
    for row in ds:
        seen += 1
        caption = (row.get('TEXT') or row.get('caption') or '').strip()
        if not caption:
            continue
        if not caption_matches(caption, regex):
            continue
        score = row.get('AESTHETIC_SCORE') or row.get('aesthetic')
        if aesthetic_min is not None:
            if score is None or score < aesthetic_min:
                continue
        url = row.get('URL') or row.get('url')
        if not url:
            continue
        yield Sample(
            occupation=occupation,
            subset='baseline' if aesthetic_min is None else f'aesthetics-{aesthetic_min}',
            image_url=url,
            caption=caption,
            aesthetic_score=score,
        )
        yielded += 1
        if yielded >= target_n:
            return
        if seen > target_n * 200:  # cap scan to avoid forever loops
            return

## Classification (DeepFace) + color features

In [ ]:
import requests, io
import numpy as np
from PIL import Image
from deepface import DeepFace

def fetch_image(url, timeout=8):
    try:
        r = requests.get(url, timeout=timeout, headers={'User-Agent': 'rar-research/0.1'})
        r.raise_for_status()
        return Image.open(io.BytesIO(r.content)).convert('RGB')
    except Exception:
        return None

def classify_one(sample):
    img = fetch_image(sample.image_url)
    if img is None:
        return
    # deepface needs file path — write to /tmp
    tmp = f'/tmp/rar_{abs(hash(sample.image_url)) & 0xffffff}.jpg'
    img.save(tmp, 'JPEG', quality=85)
    try:
        res = DeepFace.analyze(
            img_path=tmp,
            actions=['gender', 'race'],
            enforce_detection=False,
            silent=True,
        )
        r0 = res[0] if isinstance(res, list) else res
        sample.perceived_gender = r0.get('dominant_gender')
        sample.perceived_race = r0.get('dominant_race')
    except Exception:
        pass
    # color histogram in HSV space
    try:
        hsv = np.asarray(img.convert('HSV').resize((128, 128)))
        h = []
        for ch in range(3):
            cnt, _ = np.histogram(hsv[:, :, ch], bins=16, range=(0, 256))
            h.extend((cnt / cnt.sum()).tolist())
        sample.color_hist = h
    except Exception:
        pass
    finally:
        try: os.remove(tmp)
        except Exception: pass

## Hypothesis tests (H1, H2)

In [ ]:
from scipy.stats import chi2_contingency

def test_h1(baseline, filtered):
    b = Counter(s.perceived_gender for s in baseline if s.perceived_gender)
    f = Counter(s.perceived_gender for s in filtered if s.perceived_gender)
    if not b or not f:
        return None, None
    labels = sorted(set(b) | set(f))
    table = [[b.get(l, 0) for l in labels], [f.get(l, 0) for l in labels]]
    chi2, p, _, _ = chi2_contingency(table)
    n = sum(b.values()) + sum(f.values())
    cv = (chi2 / (n * (min(len(table), len(labels)) - 1))) ** 0.5
    return p, cv

def test_h2(baseline, filtered):
    b = [s.color_hist for s in baseline if s.color_hist]
    f = [s.color_hist for s in filtered if s.color_hist]
    if not b or not f:
        return None
    bv = float(np.var(np.asarray(b), axis=0).mean())
    fv = float(np.var(np.asarray(f), axis=0).mean())
    return None if bv == 0 else fv / bv

## Run the audit

This is the long-running cell. With `SMOKE=True` it should finish in ~30 minutes. With `SMOKE=False` budget 6–8 hours and don't close the tab. Kaggle's auto-save handles intermediate checkpoints in `/kaggle/working`.

In [ ]:
import pandas as pd

summary = []
started = time.time()

for cat_name in OCCUPATION_CATEGORIES:
    for canonical, regex in OCCUPATIONS[cat_name]:
        for threshold in AESTHETIC_THRESHOLDS:
            print(f'\n=== {canonical}  threshold={threshold}  N={SAMPLES_PER_SUBSET} ===')

            print(f'[{canonical}] streaming baseline…')
            baseline = list(stream_samples(
                DATASETS['baseline'], canonical, regex, SAMPLES_PER_SUBSET,
                hf_token=HF_TOKEN,
            ))
            print(f'[{canonical}] streaming aesthetic-filtered…')
            filtered = list(stream_samples(
                DATASETS['aesthetics'], canonical, regex, SAMPLES_PER_SUBSET,
                aesthetic_min=threshold, hf_token=HF_TOKEN,
            ))
            print(f'[{canonical}] sampled {len(baseline)} baseline / {len(filtered)} filtered')

            for s in tqdm(baseline, desc=f'{canonical} baseline classify'):
                classify_one(s)
            for s in tqdm(filtered, desc=f'{canonical} filtered classify'):
                classify_one(s)

            p_h1, cv = test_h1(baseline, filtered)
            var_ratio = test_h2(baseline, filtered)

            pd.DataFrame([asdict(s) for s in baseline]).to_csv(
                f'{OUT}/{canonical}_baseline.csv', index=False)
            pd.DataFrame([asdict(s) for s in filtered]).to_csv(
                f'{OUT}/{canonical}_filtered_t{threshold}.csv', index=False)

            row = {
                'occupation': canonical,
                'category': cat_name,
                'threshold': threshold,
                'n_baseline': len(baseline),
                'n_filtered': len(filtered),
                'baseline_gender': dict(Counter(s.perceived_gender for s in baseline if s.perceived_gender)),
                'filtered_gender': dict(Counter(s.perceived_gender for s in filtered if s.perceived_gender)),
                'h1_chi2_p': p_h1,
                'h1_cramers_v': cv,
                'h2_var_ratio': var_ratio,
            }
            summary.append(row)
            print(f'[{canonical}] H1 p={p_h1}  V={cv}   H2 var_ratio={var_ratio}')

            with open(f'{OUT}/summary.json', 'w') as fp:
                json.dump({
                    'mode': 'SMOKE' if SMOKE else 'FULL',
                    'started': started,
                    'elapsed_min': (time.time() - started) / 60,
                    'rows': summary,
                }, fp, indent=2, default=str)

print(f'\nDone in {(time.time() - started) / 60:.1f} min.')
print(f'Outputs in {OUT}: summary.json + per_occupation CSVs')

## Quick view of results

Run after the long cell finishes. Drop the resulting CSV / JSON into the `04_stage4_laion/results/` folder of the GitHub repo.

In [ ]:
import json, pandas as pd
with open(f'{OUT}/summary.json') as fp:
    s = json.load(fp)
df = pd.DataFrame(s['rows'])
print(f"mode={s['mode']}  elapsed={s['elapsed_min']:.1f} min")
df[['occupation', 'threshold', 'n_baseline', 'n_filtered', 'h1_chi2_p', 'h1_cramers_v', 'h2_var_ratio']]

## What to do with the results

1. **In Kaggle:** click *Output* tab → download `summary.json` and the per-occupation CSVs.
2. **Commit** to the repo at `v2/04_stage4_laion/results/`.
3. **Reply with the `summary.json` content** in chat. The paper draft (`v2/latex/main.tex` §5) gets the actual numbers inserted: H1 p-values, Cramér's V, var-ratio, plus a per-category aggregate.

If H1 holds (low p, non-trivial Cramér's V): the paper's Stage-4 claim is empirically supported; Section 5 swaps from methodology-only to methodology + result.

If H1 doesn't hold: the paper reports an honest null and reframes Stage 4 as 'the loop's curation step does not localize at LAION-Aesthetics — alternative pathways (Pinterest, RLHF reward, Common Crawl) remain.' This is also publishable, just at a different angle.